In [1]:
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import f1_score
from dataset_preprocessing import load_datasets,normalize_and_create_metadata,zero_pad_datasets,data_structure_gen

In [ ]:
datasets = load_datasets()

In [ ]:
# Data is shuffled when it is loaded into notebook
# No need to shuffle folds of data, only need to retrieve folds that are correct proportions
train_datasets, ground_truth_labels = normalize_and_create_metadata(datasets)
train_datasets = zero_pad_datasets(train_datasets)
train_data,train_labels = data_structure_gen(train_datasets)

In [ ]:
import numpy as np

def pearsons_correlation(X, Y):
    # X: (D, N, F), Y: (D, N)
    D, N, F = X.shape
    
    # Step 1: average across features per record
    X_avg = X.mean(axis=2)  # shape: (D, N)
    
    r = np.zeros(D)
    
    # Step 2: compute Pearson correlation per dataset
    for d in range(D):
        x = X_avg[d] - X_avg[d].mean()
        y = Y[d] - Y[d].mean()
        numerator = np.sum(x * y)
        denominator = np.sqrt(np.sum(x**2) * np.sum(y**2))
        r[d] = numerator / denominator
    
    return r  # shape: (D,)


In [ ]:
def mutual_information_per_dataset(X, Y):
    """
    Compute mutual information between averaged feature values and labels per dataset.
    
    X: (D, N, F)
    Y: (D, N)
    Returns: array (D,)
    """
    D, N, F = X.shape
    mi_values = np.zeros(D)

    for d in range(D):
        # mask padded features
        nonzero_mask = ~(np.all(X[d] == 0, axis=0))
        X_valid = X[d][:, nonzero_mask]

        # Step 1: summarize features for each record (mean across features)
        x_summary = X_valid.mean(axis=1).reshape(-1, 1)  # shape (N, 1)
        y_labels = Y[d]

        # Step 2: compute mutual information for this dataset
        mi = mutual_info_classif(x_summary, y_labels, discrete_features=False)
        mi_values[d] = mi[0]

    return mi_values


In [ ]:
def cosine_similarity_per_dataset(X, Y):
    """
    Computes cosine similarity between averaged feature vectors and label vectors per dataset.
    
    X: (D, N, F_max)
    Y: (D, N)
    Returns: (D,)
    """
    D, N, F = X.shape
    sims = np.zeros(D)

    for d in range(D):
        # Step 1: mask padded features
        nonzero_mask = ~(np.all(X[d] == 0, axis=0))
        X_valid = X[d][:, nonzero_mask]

        # Step 2: summarize features across records
        x_vec = X_valid.mean(axis=0)  # shape (F_d,)
        y_vec = Y[d] - Y[d].mean()    # center labels to remove bias (optional)

        # Step 3: flatten y to match (N,) or project to same dimension if needed
        # Option 1: average features per label (projection-based)
        y_vec = np.mean(X_valid[Y[d] == 1], axis=0) - np.mean(X_valid[Y[d] == 0], axis=0)

        # Step 4: cosine similarity
        num = np.dot(x_vec, y_vec)
        denom = np.linalg.norm(x_vec) * np.linalg.norm(y_vec)
        sims[d] = num / denom if denom != 0 else 0.0

    return sims


In [ ]:
ground_truth = np.array([i for i in range(0,17)])


r = pearsons_correlation(train_data,train_labels)
r_output = np.abs(r.reshape(int(r.shape[0]**0.5), int(r.shape[0]**0.5)))
r_predictions = np.argmax(r_output,axis=1)
r_f1 = f1_score(ground_truth, r_predictions, average='macro')
print(r_predictions,r_f1)

mi = mutual_information_per_dataset(train_data,train_labels)
mi_output = np.abs(r.reshape(int(mi.shape[0]**0.5), int(mi.shape[0]**0.5)))
mi_predictions = np.argmax(mi_output,axis=1)
mi_f1 = f1_score(ground_truth, mi_predictions, average='macro')
print(mi_predictions,mi_f1)

cosine = cosine_similarity_per_dataset(train_data,train_labels)
cosine_output = np.abs(r.reshape(int(cosine.shape[0]**0.5), int(cosine.shape[0]**0.5)))
cosine_predictions = np.argmax(cosine_output,axis=1)
cosine_f1 = f1_score(ground_truth, cosine_predictions, average='macro')
print(cosine_predictions,cosine_f1)